# Partial reproductions of paper Figures 1, 2, 4, 5, 6, 7, 9, 10, 11(a), and 12

Our 229 completed vision runs are compared with the authors' released measurements where timing and conditions allow. All errors are fractions. Our ResNets ran for 400 epochs versus the paper's 4,000; our SGD CNNs ran for 50,000 steps versus the paper's 500,000 (1,000,000 for CIFAR-100), and our non-augmented CIFAR-10 Adam CNNs ran for 400 versus 4,000 epochs. Every plotted cell has one seed, so our curves have no inferred error bars.

Figures 1–2 adapt the paper's 15% noise setting to our 10% run. Figure 4 now includes our CIFAR-100 ResNet as well as CIFAR-10. Figure 5 now includes both augmentation conditions. Figure 6 now compares our clean SGD and Adam arms with the authors’ corresponding published curves. The job named `figure-6` ran clean CIFAR-100 CNN SGD, so its results support the reduced Figure 7. Figures 11(a) and 12 use our 12,500/25,000 subset runs and matching full-data runs.

The authors' ResNet archive stores measurement positions that its notebook labels `index + 1`; exact epoch alignment is not independently established. For CNN plots near 50,000 steps, the authors' 256-step logging interval is an estimate from complete 1,952-point series and the 500,000-step protocol. Our `global_step` is recorded directly. Interpret side-by-side panels as reduced-horizon comparisons, not exact replications.


In [ ]:
import matplotlib.pyplot as plt
def sinkhorn(A, steps: int = 10):
    A = A / A.sum()  # optional: global mass 1
    for _ in range(steps):
        A = A / A.sum(dim=1, keepdim=True)
        A = A / A.sum(dim=0, keepdim=True)
    return A

a = torch.randn(10, 10).abs()
a = sinkhorn(a)
print(a.sum(dim=1))  # ~1
print(a.sum(dim=0))  # ~1



In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Single natural photo (not a paper figure with subpanels).
img_path = Path("../../Presentation2/figures/dino-crops/dog.jpg")
if not img_path.is_file():
    img_path = Path("../Presentation2/figures/dino-crops/dog.jpg")
img = Image.open(img_path).convert("RGB")
W, H = img.size
crop_size = 256
rng = random.Random(7)


def random_square_crop(im: Image.Image, size: int, rng: random.Random):
    w, h = im.size
    left = rng.randint(0, w - size)
    top = rng.randint(0, h - size)
    box = (left, top, left + size, top + size)
    return im.crop(box), box


crop1, box1 = random_square_crop(img, crop_size, rng)
crop2, box2 = random_square_crop(img, crop_size, rng)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
ax = axes[0]
ax.imshow(img)
for box, color, label in [(box1, "C1", "crop 1"), (box2, "C3", "crop 2")]:
    x0, y0, x1, y1 = box
    ax.add_patch(
        patches.Rectangle(
            (x0, y0), x1 - x0, y1 - y0, fill=False, linewidth=2.5, edgecolor=color
        )
    )
    ax.text(
        x0 + 6,
        y0 + 22,
        label,
        color="white",
        fontsize=10,
        fontweight="bold",
        bbox=dict(facecolor=color, edgecolor="none", pad=2),
    )
ax.set_title(f"Original {W}×{H}")
ax.axis("off")

for ax, crop, box, title in [
    (axes[1], crop1, box1, "Global crop 1 · 256×256"),
    (axes[2], crop2, box2, "Global crop 2 · 256×256"),
]:
    ax.imshow(crop)
    ax.set_title(f"{title}\n(x0,y0)=({box[0]},{box[1]})")   
    ax.axis("off")

fig.suptitle("DINOv3-style global crops (random 256×256)", fontsize=12)
plt.show()

In [ ]:
from io import BytesIO
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import Image, display

current = Path.cwd().resolve()
if (current / "our-results-folder" / "data" / "vision").is_dir():
    project_dir = current
elif (current / "Assignment 2 - Group" / "our-results-folder" / "data" / "vision").is_dir():
    project_dir = current / "Assignment 2 - Group"
elif (current.parent / "our-results-folder" / "data" / "vision").is_dir():
    project_dir = current.parent
else:
    raise FileNotFoundError("Run from ATDL-Group-Work, Assignment 2 - Group, or our-plots")
sys.path.insert(0, str(project_dir / "our-plots"))
from paper_figures import figure_1, figure_2, figure_4, figure_9, load_comparison, our_runs, WIDTHS, NOISE

paper, ours, paper_cifar100 = load_comparison(project_dir)
ours_cifar100 = our_runs(project_dir, "cifar100", "resnet", 50000, True, "adam", WIDTHS, NOISE, 400, "epoch")
print("Loaded both 30-run ResNet sweeps and the authors’ first 400 recorded points.")


def show_figure(fig: plt.Figure) -> None:
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=140, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close(fig)


## Figure 1 · model size and training dynamics

The left column shows train and test error by width at the last displayed point. The right column shows test-error curves across widths at all 40 saved time points and the oracle early-stopping envelope. Both columns transform clean test error to the noisy test distribution, as the authors’ Figure 1 code does. This uses **10% label noise**; the paper’s Figure 1 used 15%.


In [ ]:
show_figure(figure_1(paper, ours))


## Figure 2 · train and test error over width × time

Both heatmaps use **10% label noise**, the authors’ custom inferno colormap, logarithmic time, and shared color scales between the authors’ and our runs. The paper’s Figure 2 used 15%.


In [ ]:
show_figure(figure_2(paper, ours))


## Figure 4 · ResNet curves across noise levels

Both our CIFAR-10 and CIFAR-100 sweeps are shown above the authors' curves. Our runs cover 0%, 10%, and 20% noise at ten widths and epoch 400. The authors' CIFAR-10 panel also includes 5% and 15% noise. Test and train occupy separate panels.


In [ ]:
show_figure(figure_4(paper, ours, paper_cifar100, ours_cifar100))


## Figure 5 · CIFAR-10 CNN with and without augmentation

The four rows show authors and ours for augmented and non-augmented SGD through width 64. Our endpoints are at step 50,000; the authors’ position 195 is approximately step 49,920 if their unverified cadence is 256 steps per measurement.


In [ ]:
from paper_figures import (figure_5_cnn, figure_6_cnn, figure_7_cnn, figure_10_cnn,
                           figure_11a_cnn, figure_12_cnn, load_cnn_comparison,
                           load_paper_noaug, our_runs, SUBSET_WIDTHS, NOISE)
import pandas as pd
cnn_paper, cnn_ours = load_cnn_comparison(project_dir)
paper_noaug = load_paper_noaug(project_dir)
cnn_noaug = our_runs(project_dir, "cifar10", "cnn", 50000, False, "sgd", SUBSET_WIDTHS, NOISE, 50000, "global_step")
cnn_adam_noaug = our_runs(project_dir, "cifar10", "cnn", 50000, False, "adam", SUBSET_WIDTHS, NOISE, 400, "epoch")
cifar100_cnn = our_runs(project_dir, "cifar100", "cnn", 50000, False, "sgd", SUBSET_WIDTHS, (0.0,), 50000, "global_step")
subsets = pd.concat([our_runs(project_dir, "cifar10", "cnn", size, True, "sgd", SUBSET_WIDTHS, (0.1, 0.2), 50000, "global_step") for size in (12500, 25000, 50000)], ignore_index=True)
print("Loaded 48 augmented CIFAR-10 CNN, 33 non-augmented CIFAR-10 SGD CNN, 33 non-augmented CIFAR-10 Adam CNN, 11 CIFAR-100 CNN, and 44 subset runs.")


In [ ]:
show_figure(figure_5_cnn(cnn_paper, cnn_ours, paper_noaug, cnn_noaug))


## Figure 6 · clean CIFAR-10 CNN optimizer comparison

The authors’ full-horizon SGD and Adam endpoints appear above our 50,000-step SGD and 400-epoch Adam endpoints. Both local arms use the same 11 widths, clean labels, no augmentation, and one seed per cell. Different optimizer horizons and the paper’s denser grid limit direct comparison.


In [ ]:
show_figure(figure_6_cnn(project_dir, cnn_noaug, cnn_adam_noaug))


## Figure 7 · clean CIFAR-100 CNN

The authors’ five-trial endpoint mean and standard deviation appear above our one-seed result at the same 11 widths. Our horizon is 50,000 steps; the paper’s is 1,000,000.


In [ ]:
show_figure(figure_7_cnn(project_dir, cifar100_cnn))


## Figure 9 · three regimes and test-error heatmap

The left column traces widths 3, 12, and 64 with 20% noise. The right column shows the ten-width test-error surface through our epoch 400. The paper used a denser width grid and trained to epoch 4,000.


In [ ]:
show_figure(figure_9(paper, ours))


## Figure 10 · width-128 CIFAR-10 CNN histories

These are the CNN portion of Figure 10 at 0%, 10%, and 20% label noise. Both rows use optimizer steps in thousands and stop near 50,000. Our steps are recorded directly. The authors’ steps are estimated as `(measurement_index + 1) × 256`, based on the 1,952-point complete series and the paper’s 500,000-step protocol. The logging cadence remains unverified, and the authors’ 10% source has fewer recorded positions than its 0% and 20% sources. The ResNet parts of Figure 10 require separate width-128 runs.


In [ ]:
show_figure(figure_10_cnn(cnn_paper, cnn_ours))


## Figure 11(a) · CIFAR-10 CNN sample sizes

The top two rows compare published and our 20%-noise curves at the same 12,500/25,000/50,000 examples and 11 widths. Their endpoints are at the final recorded point; ours are at 50,000 steps. Our 10%-noise row is separate because the released archive contains no 10%-noise subset source.


In [ ]:
show_figure(figure_11a_cnn(project_dir, subsets))


## Figure 12 · sample size × width at 20% noise

Authors above and ours below: both heatmaps and slices use the same three sample sizes, 11 widths, and color scale. The author endpoints and our 50,000-step endpoints represent different training horizons.


In [ ]:
show_figure(figure_12_cnn(project_dir, subsets))
